# Streaming Speech-to-Speech — Moshi, Hibiki, and Full-Duplex Dialogue Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: the interface

Moshi exposes a WebSocket server that takes 80 ms chunks of Mimi-encoded audio and returns 80 ms chunks of Mimi-encoded audio. Both ways. Constantly.

In [ ]:
```python

import asyncio

import websockets

from moshi.client_utils import encode_audio_mimi, decode_audio_mimi

async def moshi_chat():

    async with websockets.connect("ws://localhost:8998/api/chat") as ws:

        mic_task = asyncio.create_task(stream_mic_to(ws))

        spk_task = asyncio.create_task(stream_from_to_speaker(ws))

        await asyncio.gather(mic_task, spk_task)

In [ ]:
```

### Step 2: the full-duplex loop

In [ ]:
```python

async def stream_mic_to(ws):

    async for chunk_80ms in mic_stream_at_12_5_hz():

        mimi_tokens = encode_audio_mimi(chunk_80ms)

        await ws.send(serialize(mimi_tokens))

async def stream_from_to_speaker(ws):

    async for msg in ws:

        mimi_tokens, text_token = deserialize(msg)

        audio = decode_audio_mimi(mimi_tokens)

        await play(audio)

In [ ]:
```

Both directions run simultaneously. Python asyncio or Rust futures are the standard transport.

### Step 3: the training objective (conceptual)

For every 80 ms frame `t`:

- Input: `user_mimi[0..t]`, `moshi_mimi[0..t-1]`, `moshi_text[0..t-1]`

- Predict: `moshi_text[t]`, then `moshi_mimi[t, codebook_0..7]`

Text is predicted before audio (inner monologue); audio is predicted codebook-sequential within the depth transformer.

### Step 4: where Moshi wins and where it doesn't

Moshi wins:

- Sub-250 ms end-to-end on cheap hardware.

- Natural back-channels and interruptions.

- No pipeline glue code.

Moshi does not win:

- Tool calling (not trained for it; you need a separate LLM path).

- Long reasoning (Moshi is an 8B-ish dialogue model, not Claude/GPT-4).

- Factual accuracy on niche topics.

- Most production enterprise use cases (still use pipelines in 2026).

## Exercises

In [ ]:
1. **Easy.** Run `code/main.py`. It simulates the two-stream + inner-monologue architecture symbolically.
2. **Medium.** Pull Moshi from HuggingFace, run the server, test one conversation. Measure wall-clock latency from end-of-user-speech to start-of-Moshi-response.
3. **Hard.** Take your Lesson 12 pipeline agent and compare P50 latency vs Moshi on 20 matched test utterances. Write up when a pipeline architecturally wins anyway.